# Data Q&A Agent - Azure AI Foundry (New Experience)

This notebook creates an Azure AI Foundry agent with Code Interpreter so you can upload CSV/Excel files and ask natural-language questions about your data.

This notebook is for the Foundry new experience only (versioned agents + Responses API), not Foundry classic threads/runs.

## Local Prerequisites

- Python 3.14.3
- Option 1 (Conda): `conda create -n sample_foundry_v2 python=3.14.3 ipykernel -y`
- Option 2 (venv): install Python 3.14, then run `py -3.14 -m venv .venv`
- `.env` file populated (copy from `.env.template`)
- `pip install -r requirements.txt`

For full from-scratch setup (including installing Python/Conda), see `README.md`.

---

## Azure Setup (run once from CLI / PowerShell)

The Foundry new experience requires a storage account connected to your project.
The Foundry project managed identity must have `Storage Blob Data Contributor` on that storage account so Code Interpreter can read/write files.

Important: Foundry resource and Foundry project have separate managed identities.
Assign storage role to the project managed identity (not only the resource identity).

```bash
# 1. Create a dedicated storage account (or use existing)
az storage account create \
  --name <STORAGE_ACCOUNT_NAME> \
  --resource-group <YOUR_RESOURCE_GROUP> \
  --location <REGION> \
  --sku Standard_LRS \
  --kind StorageV2 \
  --min-tls-version TLS1_2

# 2. Get the Foundry PROJECT managed identity principal ID
az rest --method GET \
  --url "https://management.azure.com/subscriptions/<SUB_ID>/resourceGroups/<YOUR_RESOURCE_GROUP>/providers/Microsoft.CognitiveServices/accounts/<FOUNDRY_RESOURCE_NAME>/projects/<PROJECT_NAME>?api-version=2025-04-01-preview" \
  --query "identity.principalId" -o tsv

# 3. Assign Storage Blob Data Contributor to the PROJECT identity
az role assignment create \
  --assignee-object-id <PROJECT_PRINCIPAL_ID_FROM_STEP_2> \
  --assignee-principal-type ServicePrincipal \
  --role "Storage Blob Data Contributor" \
  --scope "/subscriptions/<SUB_ID>/resourceGroups/<YOUR_RESOURCE_GROUP>/providers/Microsoft.Storage/storageAccounts/<STORAGE_ACCOUNT_NAME>"

# 4. Connect storage to Foundry project
   Go to https://ai.azure.com -> Management Center -> Connected resources
   Click "+ New connection" -> Data -> Azure Blob Storage
   Select the storage account from step 1
   Set authentication to "Microsoft Entra ID"
```

Note: Wait a few minutes for role assignment propagation before uploading files.

## 1. Install Dependencies
Run this cell once in your active environment (`sample_foundry_v2`)

OR

Follow Steps in README to install requirements in terminal

In [ ]:
%pip install -r requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


## 2. Setup & Configuration

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load .env — override=False so runtime env vars take precedence
load_dotenv(override=False)

PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.getenv("FOUNDRY_MODEL_DEPLOYMENT_NAME")

print(f"Project endpoint : {PROJECT_ENDPOINT}")
print(f"Model deployment : {MODEL_DEPLOYMENT}")

Project endpoint : https://sample-proj-resource.services.ai.azure.com/api/projects/sample_project
Model deployment : gpt-4.1


## 3. Create the Foundry Agent with Code Interpreter

The agent uses the **Code Interpreter** tool, which lets it write and execute Python code to analyse your uploaded files.

In the new Foundry experience, agents are **versioned** — each call to `create_version` publishes a new immutable version.

In [8]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    CodeInterpreterTool,
    AutoCodeInterpreterToolParam,
)
from azure.identity import DefaultAzureCredential

AGENT_NAME = "data-qa-agent-corteva"

# Create the Project client (new Foundry experience)
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

def publish_agent_version(file_ids: list[str] | None = None) -> str:
    """Publish a new immutable agent version in the Foundry new experience."""
    code_interpreter_tool = (
        CodeInterpreterTool(container=AutoCodeInterpreterToolParam(file_ids=file_ids))
        if file_ids
        else CodeInterpreterTool()
    )

    result = project_client.agents.create_version(
        agent_name=AGENT_NAME,
        definition=PromptAgentDefinition(
            model=MODEL_DEPLOYMENT,
            instructions=(
                "You are a data analyst assistant. The user will upload CSV or Excel files. "
                "Use the Code Interpreter tool to load the data with pandas, explore it, "
                "and answer the user's questions. When useful, create charts or summary tables. "
                "Always show your work by printing intermediate results."
            ),
            tools=[code_interpreter_tool],
        ),
        description="Data Q&A agent with Code Interpreter",
    )
    return result.version

# Always publish a fresh version for this notebook run (new Foundry experience).
agent_version = publish_agent_version()
print(f"✅ Published agent — Name: {AGENT_NAME}, Version: {agent_version}")

✅ Published agent — Name: data-qa-agent-corteva, Version: 1


## 4. Get the OpenAI Client

In the new Foundry experience, you interact with agents via the **OpenAI Responses API**. Multi-turn conversation is handled via `previous_response_id`.

In [11]:
openai_client = project_client.get_openai_client()

# Track the last response ID for multi-turn conversation
last_response_id = None

print("OpenAI client ready.")

OpenAI client ready.


## 5. Helper — `ask()`

A single function that handles all scenarios:
- **Just a question** — `ask("What's the total revenue?")`
- **Question + attach a file** — `ask("Summarize this", file_path="data.csv")`
- **Question + download generated files** — `ask("Add a profit column and save as Excel", download=True)`
- **All three** — `ask("Clean this and export", file_path="raw.csv", download=True)`

Uses the OpenAI Responses API with `agent_reference`. Multi-turn is handled via `previous_response_id`.

In [12]:
from pathlib import Path

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

def ask(question: str, file_path: str = None, download: bool = False) -> str:
    """Ask the agent a question via the OpenAI Responses API.

    In new Foundry agent responses, top-level `tools` is not allowed when
    `agent_reference` is specified. To attach user files for Code Interpreter,
    this function publishes a new agent version that binds uploaded file IDs
    to the Code Interpreter tool configuration.

    Args:
        question:  Your natural-language question or instruction.
        file_path: Optional path to a .csv/.xlsx file to attach to this message.
        download:  If True, download any files the agent generates to output/.
    """
    global last_response_id, agent_version

    effective_agent_version = agent_version
    file_ids = []

    # --- optionally upload a file and publish a version bound to that file ---
    if file_path:
        with open(file_path, "rb") as fp:
            uploaded = openai_client.files.create(
                file=fp,
                purpose="assistants",
            )

        print(f"📎 Uploaded: {file_path} → file ID: {uploaded.id}")
        file_ids.append(uploaded.id)

        effective_agent_version = publish_agent_version(file_ids=file_ids)
        agent_version = effective_agent_version
        last_response_id = None
        print(
            f"🆕 Published agent version with attached file context: {effective_agent_version}"
        )

    # --- call the agent via Responses API ---
    kwargs = dict(
        input=question,
        extra_body={
            "agent_reference": {
                "name": AGENT_NAME,
                "version": effective_agent_version,
                "type": "agent_reference",
            }
        },
    )

    if last_response_id:
        kwargs["previous_response_id"] = last_response_id

    response = openai_client.responses.create(**kwargs)
    last_response_id = response.id

    # --- collect response text & optionally download files ---
    response_text = ""
    for item in response.output:
        if item.type == "message":
            for block in item.content:
                if block.type == "output_text":
                    response_text += block.text
                    if download:
                        for ann in getattr(block, "annotations", []) or []:
                            if getattr(ann, "type", None) == "container_file_citation":
                                file_id = getattr(ann, "file_id", None)
                                container_id = getattr(ann, "container_id", None)
                                if not file_id or not container_id:
                                    continue

                                fname = getattr(ann, "filename", None) or f"output_{file_id}"
                                file_content = openai_client.containers.files.content.retrieve(
                                    file_id=file_id,
                                    container_id=container_id,
                                )
                                out_path = OUTPUT_DIR / fname
                                out_path.write_bytes(file_content.read())
                                print(f"⬇️  Downloaded: {out_path}")

    return response_text

## 6. Try It — All Combinations in One Thread

The cells below show every combination of `ask()`:
1. Attach a file + ask a question
2. Follow-up question (no file, no download)
3. Ask for a generated file (download=True)
4. Attach a *different* file + ask
5. Plain question with no file context

In [13]:
# 1️⃣ Attach a file + ask a question
print("=" * 60)
print("1. Attach file + question")
print("=" * 60)

answer = ask(
    "What columns are in this file? Give me a quick summary.",
    file_path="sample_data/sample_sales.csv"
)

print(answer)

1. Attach file + question
📎 Uploaded: sample_data/sample_sales.csv → file ID: assistant-S7oyARBug3qbuLWjG6mtrt
🆕 Published agent version with attached file context: 2
Here are the columns in your file:
- `date`: The date of the transaction/sale.
- `region`: The region where the sale took place.
- `product`: The product sold.
- `units_sold`: Number of units sold in each transaction.
- `unit_price`: Price per unit of the product.
- `revenue`: Total revenue from the transaction.
- `cost`: Total cost associated with the transaction.

Quick summary:
- The file contains 20 records.
- There are 4 unique regions and 3 unique products.
- Units sold range from 45 to 155, with an average of about 97.
- Unit price ranges from 25 to 55, averaging about 38.
- Revenue per transaction ranges from 2200 to 5000.
- Cost per transaction ranges from 1320 to 3125.

If you'd like more detailed breakdowns (such as by region or product), let me know!


In [14]:
# 2️⃣ Follow-up — no file, no download (agent remembers the CSV)
print("=" * 60)
print("2. Follow-up question (no file)")
print("=" * 60)
answer = ask("Which region has the highest total revenue?")
print(answer)

2. Follow-up question (no file)
The region with the highest total revenue is North, with a total revenue of 20,400. Here is the total revenue by region:

- North: 20,400
- South: 16,625
- East: 16,400
- West: 13,800

If you want a visualization or further breakdown, let me know!


In [15]:
# 3️⃣ Ask to generate a modified file → downloads to output/
print("=" * 60)
print("3. Generate + download a modified file")
print("=" * 60)
answer = ask(
    "Add a 'profit' column (revenue minus cost) and a 'profit_margin_pct' column. "
    "Save the result as a new Excel file.",
    download=True
)
print(answer)

3. Generate + download a modified file
⬇️  Downloaded: output\sales_with_profit.xlsx
I've added the 'profit' and 'profit_margin_pct' columns to your data. You can download the updated file here:

[Download the updated Excel file](sandbox:/mnt/data/sales_with_profit.xlsx)


In [16]:
# 3️⃣ Ask to generate a modified file → downloads to output/
print("=" * 60)
print("3. Generate + download a modified file")
print("=" * 60)
answer = ask(
    "add another column that indicates whether profit_margin_pct is >= 35 (1) or < (0)."
    "Save the result as a new Excel file.",
    download=True
)
print(answer)

3. Generate + download a modified file
⬇️  Downloaded: output\sales_with_profit_margin_flag.xlsx
I've added the requested column, `high_profit_margin`, which is set to 1 when the profit margin percentage is 35 or higher, and 0 otherwise.

You can download the updated file here:

[Download the Excel file with profit margin flag](sandbox:/mnt/data/sales_with_profit_margin_flag.xlsx)


## 7. Cleanup

Delete the agent when you're done. (No threads to clean up in the new Foundry experience.)

In [ ]:
project_client.agents.delete(agent_name=AGENT_NAME)
print(f"Agent '{AGENT_NAME}' deleted.")